In [0]:
%sql
-- ============================================================
--  Crear el esquema Landing
-- ============================================================

--Crear catalogo si no existe
create catalog real_estate;
-- Usar tu catálogo
use catalog real_estate;

In [0]:
%sql
-- Crear esquema Landing

create schema if not exists real_estate.landing
comment 'Esquemas para archivos';

In [0]:
%sql
-- Crear esquema Raw

create schema if not exists real_estate.bronze
comment 'Esquemas para datos crudos sin procesar';

In [0]:
%sql
-- Verificar

show schemas;

In [0]:
%sql
-- ============================================================
-- Crear el Volume para archivos
-- ============================================================

-- Crear volume tipo MANAGED (Databricks administra el storage)
create volume if not exists real_estate.landing.archivos
comment 'Volume para almacenar archivos CSV crudos';

In [0]:
%sql
--vericar que se creo 
show volumes in real_estate.landing;

In [0]:
%sql
-- ============================================================
-- Verificar que el archivo se subió correctamente
-- ============================================================

--Listar archivos en volume
list '/Volumes/real_estate/landing/archivos'

In [0]:
%sql
-- Crear la tabla properties_bronze

-- Leer directo del archivo es posible, y es útil también para analizarlo

select * from read_files (
    '/Volumes/real_estate/landing/archivos',
    format => 'csv',
    header => true 
)

In [0]:
%sql

-- ============================================================
-- Crear la tabla properties_bronze
-- ============================================================

-- Primero, eliminamos la tabla si existe (para poder recrearla)
drop table if exists real_estate.bronze.properties_bronze;

-- Crear tabla EXTERNA leyendo el CSV y nos quedamos solo con los registros que tienen url válida
create table real_estate.bronze.properties_bronze
select * from read_files (
    '/Volumes/real_estate/landing/archivos',
    format => 'csv',
    header => true
)
where url like 'https%' --Filtramos por aquellos links que sean consistentes
;

In [0]:
%sql
-- Verificar que la tabla se creó correctamente
show tables in real_estate.bronze;

In [0]:
%sql
use catalog real_estate;
show schemas;

In [0]:
%sql
select 
    count(*) as total_registros,
    count(distinct zona) as zona_unicas,
    count(distinct tipo_de_operacion) as tipos_operacion
from real_estate.bronze.properties_bronze

In [0]:
%sql
create schema if not exists real_estate.silver
comment 'Capa Silver: Datos limpios, validados y con tipos correctos';

describe schema real_estate.silver;

In [0]:
%sql

create schema if not exists real_estate.gold
comment 'Capa Gold: Modelo para el negocio sobre el cuál realizar calculos de métricas';

describe schema real_estate.gold;

In [0]:
%sql
show schemas in real_estate;